In [ ]:
%pip install -q wandb huggingface_hub
%pip install -q datasets
%pip install -q transformers
%pip install -q deepspeed

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import HfApi
from datasets import load_dataset
import torch
import numpy as np
import random
import json
import uuid
import deepspeed
import shutil

from deepspeed import comm as ds_dist
from deepspeed.moe.layer import MoE

import math
import time
from contextlib import nullcontext
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tqdm.auto import tqdm


In [ ]:
assert torch.cuda.is_available()

torch.cuda.set_device(0)
os.environ.setdefault("LOCAL_RANK", "0")

if not ds_dist.is_initialized():
    deepspeed.init_distributed(
        dist_backend="nccl",
        auto_mpi_discovery=False,
        init_method=f"file:///tmp/moe_init_{uuid.uuid4().hex}",
        rank=0,
        world_size=1,
    )

assert ds_dist.get_world_size() == 1
print("DeepSpeed:", deepspeed.__version__)

In [3]:
secrets = UserSecretsClient()

wandb.login(key=secrets.get_secret("wandb"))

hf_api = HfApi(token=secrets.get_secret("huggingface-cli"))
print("HF account:", hf_api.whoami()["name"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zvil (mrzvil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


HF account: MrZVIL


In [4]:
os.environ['HF_HOME'] = '/kaggle/working/hf_hub'

In [ ]:
experiment_name = "prenorm_rmsnorm_swiglu_relative_moe_encdec"

purpose = (
    "Continue training the existing MoE Transformer on 10M "
    "training examples for one additional epoch, preserving "
    "AdamW state and using a new cosine schedule from 5e-5 "
    "to zero without warmup."
)

run = wandb.init(
    project="hw2-code-generation",
    name=f"{experiment_name}_stage2_10m",
    config={"purpose": purpose},
)

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def capture_rng_state():
    np_state = np.random.get_state()

    return {
        "python": random.getstate(),
        "numpy": {
            "name": np_state[0],
            "keys": np_state[1].tolist(),
            "position": np_state[2],
            "has_gauss": np_state[3],
            "cached_gaussian": np_state[4],
        },
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }


def restore_rng_state(state):
    random.setstate(state["python"])

    np_state = state["numpy"]
    np.random.set_state((
        np_state["name"],
        np.asarray(np_state["keys"], dtype=np.uint32),
        np_state["position"],
        np_state["has_gauss"],
        np_state["cached_gaussian"],
    ))

    torch.set_rng_state(state["torch"])
    torch.cuda.set_rng_state_all(state["cuda"])

SEED = 67
set_global_seed(SEED)

In [7]:
hf_user_name = hf_api.whoami()["name"]
repo_id = f"{hf_user_name}/hw2-cg-{experiment_name}"

hf_api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
)

RepoUrl('https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer', endpoint='https://huggingface.co', repo_type='model', repo_id='MrZVIL/hw2-cg-baseline_transformer')

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

checkpoint_prefix = "training/moe_40m"

download_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    token=secrets.get_secret("huggingface-cli"),
    allow_patterns=[
        f"{checkpoint_prefix}/last.pt",
        f"{checkpoint_prefix}/best.pt",
        f"{checkpoint_prefix}/tokenizer.json",
    ],
)

resume_folder = Path(download_dir) / checkpoint_prefix

resume_checkpoint = torch.load(
    resume_folder / "last.pt",
    map_location="cpu",
    weights_only=True,
)

In [ ]:
config = dict(resume_checkpoint["config"])

config.update({
    "train_size": 10_000_000,
    "epochs": 1,
    "learning_rate": 5e-5,
    "warmup_ratio": 0.0,
    "training_stage": "continued_training",
    "parent_global_step": resume_checkpoint["global_step"],
})

tokenizer = Tokenizer.from_file(
    str(resume_folder / "tokenizer.json")
)

In [8]:
dataset = load_dataset("SlavaYus/tinysms_qa_12m")
print(dataset)

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'code'],
        num_rows: 11838003
    })
    test: Dataset({
        features: ['question', 'code'],
        num_rows: 1919
    })
})


In [ ]:
split = dataset["train"].train_test_split(
    test_size=0.10,
    shuffle=True,
    seed=config['seed']
)

train_dataset = split["train"]
valid_dataset = split["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 10654202
Valid: 1183801
Test: 1919


In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)


Device: cuda


In [12]:
train_raw = train_dataset.select(
    range(min(config["train_size"], len(train_dataset)))
)
valid_raw = valid_dataset.select(
    range(min(config["valid_size"], len(valid_dataset)))
)

In [ ]:
PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

assert all(
    value is not None
    for value in [PAD_ID, BOS_ID, EOS_ID]
)
assert PAD_ID == resume_checkpoint["pad_id"]
assert tokenizer.get_vocab_size() == config["vocab_size"]

example = train_raw[0]["code"]
assert tokenizer.decode(tokenizer.encode(example).ids) == example

print("Vocabulary:", tokenizer.get_vocab_size())
print(
    "Resume position:",
    resume_checkpoint["epoch"],
    resume_checkpoint["batch_idx"],
    resume_checkpoint["global_step"],
)




Vocabulary: 16000


In [14]:
def tokenize_batch(batch):
    questions = tokenizer.encode_batch(batch["question"])
    codes = tokenizer.encode_batch(batch["code"])

    return {
        "src_ids": [
            x.ids[:config["max_src_len"] - 1] + [EOS_ID]
            for x in questions
        ],
        "tgt_ids": [
            [BOS_ID] + x.ids[:config["max_tgt_len"] - 2] + [EOS_ID]
            for x in codes
        ],
        "src_truncated": [
            len(x.ids) > config["max_src_len"] - 1 for x in questions
        ],
        "tgt_truncated": [
            len(x.ids) > config["max_tgt_len"] - 2 for x in codes
        ],
    }

train_encoded = train_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_raw.column_names,
    desc="Tokenize train",
)
valid_encoded = valid_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=valid_raw.column_names,
    desc="Tokenize valid",
)

data_stats = {
    "data/train_size": len(train_raw),
    "data/valid_size": len(valid_raw),
}

for name, data in [("train", train_encoded), ("valid", valid_encoded)]:
    for field in ["src_truncated", "tgt_truncated"]:
        data_stats[f"data/{name}_{field}"] = float(np.mean(data[field]))

print(data_stats)

def collate_fn(examples):
    src = pad_sequence(
        [torch.tensor(x["src_ids"], dtype=torch.long) for x in examples],
        batch_first=True,
        padding_value=PAD_ID,
    )
    tgt = pad_sequence(
        [torch.tensor(x["tgt_ids"], dtype=torch.long) for x in examples],
        batch_first=True,
        padding_value=PAD_ID,
    )
    return src, tgt

loader_generator = torch.Generator().manual_seed(config["seed"])

train_loader = DataLoader(
    train_encoded,
    batch_size=config["batch_size"],
    shuffle=True,
    generator=loader_generator,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
valid_loader = DataLoader(
    valid_encoded,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

Tokenize train:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenize valid:   0%|          | 0/20000 [00:00<?, ? examples/s]

{'data/train_size': 500000, 'data/valid_size': 20000, 'data/train_src_truncated': 2e-06, 'data/train_tgt_truncated': 0.003262, 'data/valid_src_truncated': 0.0, 'data/valid_tgt_truncated': 0.00295}


In [ ]:
class BasicFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.linear1 = nn.Linear(
            config['d_model'],
            config['dim_feedforward'],
        )
        self.linear2 = nn.Linear(
            config['dim_feedforward'],
            config['d_model'],
        )
        self.dropout = nn.Dropout(config['dropout'])
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class SwiGLUFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]
        hidden_dim = config["dim_feedforward"]

        self.gate_proj = nn.Linear(d_model, hidden_dim)
        self.up_proj = nn.Linear(d_model, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        hidden = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(self.dropout(hidden))

class DenseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ffn = SwiGLUFeedForward(config)

    def forward(self, x, padding_mask=None):
        output = self.ffn(x)
        aux_loss = output.new_zeros((), dtype=torch.float32)
        return output, aux_loss, None

In [ ]:
class MoEFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_experts = config["num_experts"]

        self.moe = MoE(
            hidden_size=config["d_model"],
            expert=SwiGLUFeedForward(config),
            num_experts=self.num_experts,
            ep_size=1,
            k=config["moe_top_k"],
            drop_tokens=False,
            use_residual=False,
            noisy_gate_policy=None,
            top2_2nd_expert_sampling=False,
            use_tutel=False,
        )

        self.moe.set_deepspeed_parallelism()

    def forward(self, x, padding_mask=None):
        flat = x.reshape(-1, x.size(-1))

        valid = (
            torch.ones(
                flat.size(0),
                dtype=torch.bool,
                device=x.device,
            )
            if padding_mask is None
            else ~padding_mask.reshape(-1)
        )

        indices = valid.nonzero(as_tuple=True)[0]
        tokens = flat[indices]

        if tokens.size(0) == 0:
            return (
                torch.zeros_like(x),
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros(self.num_experts, dtype=torch.float32),
            )

        output, aux_loss, counts = self.moe(tokens)

        result = torch.zeros_like(flat).index_copy(
            0,
            indices,
            output.to(flat.dtype),
        )

        counts = counts.detach().to(
            device=x.device,
            dtype=torch.float32,
        )
        load = counts / counts.sum().clamp_min(1)

        return result.view_as(x), aux_loss, load

In [ ]:
class RelativePositionBias(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.max_distance = config["max_relative_distance"]
        self.num_heads = config["nhead"]

        self.embedding = nn.Embedding(
            2 * self.max_distance + 1,
            self.num_heads,
        )

    def forward(self, x, padding_mask=None, causal_mask=None):
        batch_size, length, _ = x.shape
        positions = torch.arange(length, device=x.device)

        distances = positions[None, :] - positions[:, None]
        indices = (
            distances.clamp(-self.max_distance, self.max_distance)
            + self.max_distance
        )

        bias = self.embedding(indices).permute(2, 0, 1)
        bias = bias.to(dtype=x.dtype)
        bias = bias.unsqueeze(0).expand(
            batch_size, -1, -1, -1
        )

        if padding_mask is not None:
            bias = bias.masked_fill(
                padding_mask[:, None, None, :],
                float("-inf"),
            )

        if causal_mask is not None:
            bias = bias.masked_fill(
                causal_mask[None, None, :, :],
                float("-inf"),
            )

        return bias.reshape(
            batch_size * self.num_heads,
            length,
            length,
        )

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )
        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.dropout1 = nn.Dropout(config['dropout'])
        self.dropout2 = nn.Dropout(config['dropout'])

    def attention(self, x, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def forward(self, x, padding_mask=None):
        x = x + self.dropout1(
            self.attention(self.norm1(x), padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm2(x),
            padding_mask=padding_mask,
        )

        x = x + self.dropout2(ffn_output)

        return x, aux_loss, expert_load

class DecoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )

        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm3 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.dropout1 = nn.Dropout(config["dropout"])
        self.dropout2 = nn.Dropout(config["dropout"])
        self.dropout3 = nn.Dropout(config["dropout"])

    def self_attention(self, x, causal_mask, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
            causal_mask=causal_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def cross_attention(self, x, memory, memory_padding_mask):
        output, _ = self.cross_attn(
            query=x,
            key=memory,
            value=memory,
            key_padding_mask=memory_padding_mask,
            need_weights=False,
        )
        return output

    def forward(
        self,
        x,
        memory,
        causal_mask,
        padding_mask=None,
        memory_padding_mask=None,
    ):
        x = x + self.dropout1(
            self.self_attention(self.norm1(x), causal_mask, padding_mask)
        )
        x = x + self.dropout2(
            self.cross_attention(self.norm2(x), memory, memory_padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm3(x),
            padding_mask=padding_mask,
        )
        x = x + self.dropout3(ffn_output)

        return x, aux_loss, expert_load

In [ ]:
class CodeTransformer(nn.Module):
    def __init__(self, config, pad_id):
        super().__init__()

        self.pad_id = pad_id
        self.d_model = config["d_model"]

        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            self.d_model,
            padding_idx=pad_id,
        )
        self.num_experts = config["num_experts"]
        self.max_src_len = config["max_src_len"]
        self.max_tgt_len = config["max_tgt_len"]
        
        self.embedding_dropout = nn.Dropout(config["dropout"])

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                config,
                use_moe=i in config["encoder_moe_layers"],
            )
            for i in range(config["num_encoder_layers"])
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                config,
                use_moe=i in config["decoder_moe_layers"],
            )
            for i in range(config["num_decoder_layers"])
        ])

        self.encoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.decoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.output = nn.Linear(
            self.d_model,
            config["vocab_size"],
        )

        self.reset_parameters()

    def reset_parameters(self):
        for layers in (self.encoder_layers, self.decoder_layers):
            for parameter in layers.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)

        nn.init.normal_(self.token_embedding.weight, std=0.02)

        for module in self.modules():
            if isinstance(module, RelativePositionBias):
                nn.init.zeros_(module.embedding.weight)

        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()

    def collect_aux(self, x, aux_losses, expert_loads):
        if not aux_losses:
            return (
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros((0, self.num_experts), dtype=torch.float32),
            )

        return (
            torch.stack(aux_losses).mean(),
            torch.stack(expert_loads),
        )

    def embed(self, ids):
        x = self.token_embedding(ids) * math.sqrt(self.d_model)
        return self.embedding_dropout(x)

    def encode(self, src, return_aux=False):
        padding_mask = src.eq(self.pad_id)
        memory = self.embed(src)

        aux_losses = []
        expert_loads = []

        for layer in self.encoder_layers:
            memory, aux_loss, expert_load = layer(
                memory,
                padding_mask=padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        memory = self.encoder_norm(memory)

        if return_aux:
            aux_loss, loads = self.collect_aux(
                memory, aux_losses, expert_loads
            )
            return memory, aux_loss, loads

        return memory


    def decode(
        self,
        tgt_input,
        memory,
        memory_padding_mask=None,
        return_aux=False,
    ):
        length = tgt_input.size(1)

        causal_mask = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=tgt_input.device,
            ),
            diagonal=1,
        )

        padding_mask = tgt_input.eq(self.pad_id)
        x = self.embed(tgt_input)

        aux_losses = []
        expert_loads = []

        for layer in self.decoder_layers:
            x, aux_loss, expert_load = layer(
                x,
                memory,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                memory_padding_mask=memory_padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        logits = self.output(self.decoder_norm(x))

        if return_aux:
            aux_loss, loads = self.collect_aux(
                x, aux_losses, expert_loads
            )
            return logits, aux_loss, loads

        return logits


    def forward(self, src, tgt_input, return_aux=False):
        if not return_aux:
            memory = self.encode(src)
            return self.decode(
                tgt_input,
                memory,
                memory_padding_mask=src.eq(self.pad_id),
            )

        memory, encoder_aux, encoder_loads = self.encode(
            src,
            return_aux=True,
        )

        logits, decoder_aux, decoder_loads = self.decode(
            tgt_input,
            memory,
            memory_padding_mask=src.eq(self.pad_id),
            return_aux=True,
        )

        n_encoder = encoder_loads.size(0)
        n_decoder = decoder_loads.size(0)

        aux_loss = (
            encoder_aux * n_encoder + decoder_aux * n_decoder
        ) / max(n_encoder + n_decoder, 1)

        expert_loads = torch.cat(
            [encoder_loads, decoder_loads],
            dim=0,
        )

        return logits, aux_loss, expert_loads

In [16]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    token_count = 0

    for src, tgt in loader:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        labels = tgt[:, 1:]

        with amp_context():
            logits = model(src, tgt[:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum",
            )

        mask = labels.ne(PAD_ID)
        loss_sum += loss.item()
        correct += ((logits.argmax(-1) == labels) & mask).sum().item()
        token_count += mask.sum().item()

    return {
        "val/loss": loss_sum / token_count,
        "val/token_accuracy": correct / token_count,
    }

In [ ]:
save_checkpoint_prefix = "training/moe_40m_stage2"


def upload_training_checkpoint(output_dir):
    hf_api.upload_folder(
        repo_id=repo_id,
        repo_type="model",
        folder_path=str(output_dir),
        path_in_repo=save_checkpoint_prefix,
        allow_patterns=[
            "last.pt",
            "best.pt",
            "tokenizer.json",
        ],
        commit_message="Save stage 2 checkpoint",
    )

In [ ]:
use_amp = device.type == "cuda"

def amp_context():
    return (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if use_amp else nullcontext()
    )

def train(
    model,
    train_loader,
    valid_loader,
    optimizer,
    config,
    device,
    pad_id,
    run,
    output_dir,
    scheduler=None,
    resume_checkpoint=None,
    checkpoint_callback=None,
    checkpoint_every_minutes=30,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    previous_best = resume_folder / "best.pt"

    if previous_best.exists():
        shutil.copy2(previous_best, output_dir / "best.pt")

    if train_loader.generator is None:
        raise ValueError("train_loader должен иметь отдельный generator")

    if train_loader.num_workers != 0:
        raise ValueError("Этот вариант resume рассчитан на num_workers=0")

    if len(train_loader) == 0:
        raise ValueError("train_loader пуст")

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )

    global_step = 0
    best_val_loss = float("inf")
    history = []
    start_epoch = 1
    completed_batches = 0

    moe_layer_names = [
        ("encoder", i)
        for i, layer in enumerate(model.encoder_layers)
        if layer.use_moe
    ] + [
        ("decoder", i)
        for i, layer in enumerate(model.decoder_layers)
        if layer.use_moe
    ]

    def capture_rng_state():
        np_state = np.random.get_state()

        return {
            "python": random.getstate(),
            "numpy": {
                "name": np_state[0],
                "keys": np_state[1].tolist(),
                "position": np_state[2],
                "has_gauss": np_state[3],
                "cached_gaussian": np_state[4],
            },
            "torch": torch.get_rng_state(),
            "cuda": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else []
            ),
        }

    def restore_rng_state(state):
        random.setstate(state["python"])

        np_state = state["numpy"]
        np.random.set_state((
            np_state["name"],
            np.asarray(np_state["keys"], dtype=np.uint32),
            np_state["position"],
            np_state["has_gauss"],
            np_state["cached_gaussian"],
        ))

        torch.set_rng_state(state["torch"].cpu())

        if state["cuda"] and torch.cuda.is_available():
            torch.cuda.set_rng_state_all([
                value.cpu() for value in state["cuda"]
            ])

    def atomic_save(payload, filename):
        path = output_dir / filename
        temporary_path = output_dir / f"{filename}.tmp"

        torch.save(payload, temporary_path)
        os.replace(temporary_path, path)

    def save_last(
        epoch,
        batch_idx,
        epoch_generator_state,
        epoch_loss_sum,
        epoch_tokens,
    ):
        atomic_save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": (
                scheduler.state_dict()
                if scheduler is not None
                else None
            ),
            "scaler_state_dict": scaler.state_dict(),
            "config": dict(config),
            "pad_id": pad_id,
            "epoch": epoch,
            "batch_idx": batch_idx,
            "global_step": global_step,
            "best_val_loss": best_val_loss,
            "epoch_loss_sum": epoch_loss_sum,
            "epoch_tokens": epoch_tokens,
            "epoch_generator_state": epoch_generator_state,
            "rng_state": capture_rng_state(),
            "history": history,
            "wandb_run_id": run.id,
        }, "last.pt")

        if checkpoint_callback is not None:
            checkpoint_callback(output_dir)

    def validate(epoch):
        nonlocal best_val_loss

        metrics = evaluate(model, valid_loader)
        model.train()

        if not np.isfinite(metrics["val/loss"]):
            raise RuntimeError("Validation loss became NaN or Inf")

        if metrics["val/loss"] < best_val_loss:
            best_val_loss = metrics["val/loss"]

            atomic_save({
                "model_state_dict": model.state_dict(),
                "config": dict(config),
                "pad_id": pad_id,
                "epoch": epoch,
                "global_step": global_step,
                "val_loss": best_val_loss,
                "wandb_run_id": run.id,
            }, "best.pt")

            run.summary["best/val_loss"] = best_val_loss
            run.summary["best/epoch"] = epoch

        return metrics

    if resume_checkpoint is not None:
        saved_config = resume_checkpoint["config"]

        checked_keys = [
            "seed",
            "train_size",
            "valid_size",
            "batch_size",
            "epochs",
            "vocab_size",
            "max_src_len",
            "max_tgt_len",
            "d_model",
            "nhead",
            "num_encoder_layers",
            "num_decoder_layers",
            "dim_feedforward",
            "encoder_moe_layers",
            "decoder_moe_layers",
            "num_experts",
            "moe_top_k",
            "moe_aux_coef",
            "learning_rate",
            "weight_decay",
            "scheduler",
            "warmup_ratio",
            "num_training_steps",
            "num_warmup_steps",
        ]

        for key in checked_keys:
            if saved_config.get(key) != config.get(key):
                raise ValueError(
                    f"Config mismatch: {key}: "
                    f"{saved_config.get(key)} != {config.get(key)}"
                )

        if resume_checkpoint["pad_id"] != pad_id:
            raise ValueError("PAD_ID отличается от checkpoint")

        saved_scheduler = resume_checkpoint["scheduler_state_dict"]

        if (scheduler is None) != (saved_scheduler is None):
            raise ValueError("Scheduler отличается от checkpoint")

        model.load_state_dict(
            resume_checkpoint["model_state_dict"]
        )
        optimizer.load_state_dict(
            resume_checkpoint["optimizer_state_dict"]
        )

        if scheduler is not None:
            scheduler.load_state_dict(saved_scheduler)

        scaler.load_state_dict(
            resume_checkpoint["scaler_state_dict"]
        )

        global_step = resume_checkpoint["global_step"]
        best_val_loss = resume_checkpoint["best_val_loss"]
        history = list(resume_checkpoint["history"])
        start_epoch = resume_checkpoint["epoch"]
        completed_batches = resume_checkpoint["batch_idx"]

        if not 0 <= completed_batches <= len(train_loader):
            raise ValueError("Некорректная позиция в train_loader")

        run.summary["best/val_loss"] = best_val_loss

        print(
            f"Resume: epoch={start_epoch}, "
            f"completed_batches={completed_batches}, "
            f"global_step={global_step}"
        )

    if start_epoch > config["epochs"]:
        print("Все запланированные эпохи уже завершены")
        return history

    last_checkpoint_time = time.monotonic()

    for epoch in range(start_epoch, config["epochs"] + 1):
        model.train()

        resuming_epoch = (
            resume_checkpoint is not None
            and epoch == start_epoch
        )

        if resuming_epoch:
            epoch_loss_sum = resume_checkpoint["epoch_loss_sum"]
            epoch_tokens = resume_checkpoint["epoch_tokens"]
            epoch_generator_state = resume_checkpoint[
                "epoch_generator_state"
            ].cpu()

            train_loader.generator.set_state(
                epoch_generator_state
            )
            skip_batches = completed_batches
        else:
            epoch_loss_sum = 0.0
            epoch_tokens = 0
            skip_batches = 0
            epoch_generator_state = (
                train_loader.generator.get_state().clone()
            )

        iterator = iter(train_loader)

        if skip_batches:
            for _ in tqdm(
                range(skip_batches),
                desc="Skipping completed batches",
            ):
                next(iterator)

        if resuming_epoch:
            restore_rng_state(resume_checkpoint["rng_state"])

        window_loss_sum = 0.0
        window_tokens = 0
        window_start = time.perf_counter()

        progress = tqdm(
            iterator,
            total=len(train_loader),
            initial=skip_batches,
            desc=f"Epoch {epoch}",
        )

        for batch_idx, (src, tgt) in enumerate(
            progress,
            start=skip_batches + 1,
        ):
            src = src.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)

            tgt_input = tgt[:, :-1]
            labels = tgt[:, 1:]
            n_tokens = labels.ne(pad_id).sum().item()

            optimizer.zero_grad(set_to_none=True)

            with amp_context():
                logits, moe_aux_loss, expert_loads = model(
                    src,
                    tgt_input,
                    return_aux=True,
                )

                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1),
                    ignore_index=pad_id,
                )

                total_loss = (
                    loss + config["moe_aux_coef"] * moe_aux_loss
                )

            if not torch.isfinite(total_loss).item():
                raise RuntimeError("Loss became NaN or Inf")

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)

            grad_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                config["grad_clip"],
            )

            lr_used = optimizer.param_groups[0]["lr"]

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            step_skipped = scaler.get_scale() < old_scale

            if not step_skipped:
                global_step += 1

                if scheduler is not None:
                    scheduler.step()

            batch_loss_sum = loss.item() * n_tokens

            epoch_loss_sum += batch_loss_sum
            epoch_tokens += n_tokens
            window_loss_sum += batch_loss_sum
            window_tokens += n_tokens

            should_log = (
                batch_idx % config["log_every"] == 0
                or batch_idx == len(train_loader)
            )

            if should_log:
                elapsed = time.perf_counter() - window_start
                window_loss = window_loss_sum / window_tokens

                metrics = {
                    "global_step": global_step,
                    "epoch": epoch,
                    "train/loss": window_loss,
                    "train/lr": lr_used,
                    "train/grad_norm": grad_norm.item(),
                    "train/target_tokens_per_second": (
                        window_tokens / elapsed
                    ),
                    "train/moe_aux_loss_last_batch": (
                        moe_aux_loss.detach().item()
                    ),
                    "train/total_loss_last_batch": (
                        total_loss.detach().item()
                    ),
                }

                loads = expert_loads.detach().cpu().tolist()

                for (stack, layer_idx), layer_loads in zip(
                    moe_layer_names, loads
                ):
                    for expert_idx, value in enumerate(layer_loads):
                        metrics[
                            f"moe/{stack}_{layer_idx}/"
                            f"expert_{expert_idx}"
                        ] = value

                run.log(metrics)
                progress.set_postfix(loss=window_loss)

                window_loss_sum = 0.0
                window_tokens = 0
                window_start = time.perf_counter()

            should_valid = (
                batch_idx % config["valid_every"] == 0
                and batch_idx != len(train_loader)
            )

            if should_valid:
                pause_start = time.perf_counter()

                metrics = validate(epoch)
                run.log({
                    **metrics,
                    "global_step": global_step,
                    "epoch": epoch,
                })

                window_start += time.perf_counter() - pause_start

            checkpoint_due = (
                time.monotonic() - last_checkpoint_time
                >= checkpoint_every_minutes * 60
            )

            if checkpoint_due and batch_idx != len(train_loader):
                pause_start = time.perf_counter()

                save_last(
                    epoch,
                    batch_idx,
                    epoch_generator_state,
                    epoch_loss_sum,
                    epoch_tokens,
                )

                last_checkpoint_time = time.monotonic()
                window_start += time.perf_counter() - pause_start

        metrics = validate(epoch)
        metrics.update({
            "global_step": global_step,
            "epoch": epoch,
            "train/epoch_loss": epoch_loss_sum / epoch_tokens,
        })

        run.log(metrics)
        history.append(metrics)
        print(f"Epoch {epoch}: {metrics}")

        save_last(
            epoch=epoch + 1,
            batch_idx=0,
            epoch_generator_state=(
                train_loader.generator.get_state().clone()
            ),
            epoch_loss_sum=0.0,
            epoch_tokens=0,
        )

        last_checkpoint_time = time.monotonic()
        resume_checkpoint = None

    return history

In [ ]:
set_global_seed(config["seed"])

model = CodeTransformer(config, PAD_ID).to(device)
model.load_state_dict(resume_checkpoint["model_state_dict"])
model.compile()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

optimizer.load_state_dict(
    resume_checkpoint["optimizer_state_dict"]
)

for group in optimizer.param_groups:
    group["lr"] = config["learning_rate"]
    group["initial_lr"] = config["learning_rate"]

num_training_steps = len(train_loader) * config["epochs"]
num_warmup_steps = 0

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

config.update({
    "num_training_steps": num_training_steps,
    "num_warmup_steps": num_warmup_steps,
})

stage_checkpoint = {
    **resume_checkpoint,
    "config": dict(config),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "epoch": 1,
    "batch_idx": 0,
    "epoch_loss_sum": 0.0,
    "epoch_tokens": 0,
    "history": [],
}

output_dir = Path("/kaggle/working/baseline") / run.id
output_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save(str(output_dir / "tokenizer.json"))

previous_best = resume_folder / "best.pt"

if previous_best.exists():
    shutil.copy2(previous_best, output_dir / "best.pt")
else:
    stage_checkpoint["best_val_loss"] = float("inf")

run.config.update(config)
run.summary.update(data_stats)
run.summary["model/parameters"] = sum(
    p.numel() for p in model.parameters()
)
run.summary["device"] = str(device)
run.summary["torch_version"] = str(torch.__version__)

print("Parameters:", run.summary["model/parameters"])
print("Starting LR:", optimizer.param_groups[0]["lr"])
print("Additional training steps:", num_training_steps)

Parameters: 50179712


In [ ]:
history = train(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    config=config,
    device=device,
    pad_id=PAD_ID,
    run=run,
    output_dir=output_dir,
    scheduler=scheduler,
    checkpoint_callback=upload_training_checkpoint,
    resume_checkpoint=stage_checkpoint,
    checkpoint_every_minutes=30,
)

Epoch 1:   0%|          | 0/15625 [00:00<?, ?it/s]

W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] Graph break from `Tensor.item()`, consider setting:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] or:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] to include these operations in the captured graph.
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] 
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0] Graph break: from user code at:
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]   File "/tmp/ipykernel_1204/3936584717.py", line 50, in forward
W1013 10:40:22.191000 1204 torch/_dynamo/variables/tensor.py:1379] [0/0]     hidden = self.transformer(
W1013 10

Epoch 1: {'val/loss': 0.6744380075596893, 'val/token_accuracy': 0.8371618661606852, 'global_step': 15622, 'epoch': 1, 'train/epoch_loss': 1.1189774991942043}


Epoch 2:   0%|          | 0/15625 [00:00<?, ?it/s]

: 

In [ ]:
@torch.inference_mode()
def generate_batch(
    model,
    tokenizer,
    questions,
    max_src_len,
    max_new_tokens=64,
):
    if not questions:
        return []

    device = next(model.parameters()).device
    model.eval()

    encoded = tokenizer.encode_batch(questions)
    src = pad_sequence(
        [
            torch.tensor(
                item.ids[:max_src_len - 1] + [EOS_ID],
                dtype=torch.long,
            )
            for item in encoded
        ],
        batch_first=True,
        padding_value=PAD_ID,
    ).to(device)

    src_padding = src.eq(PAD_ID)
    memory = model.encode(src)

    generated = torch.full(
        (len(questions), 1),
        BOS_ID,
        dtype=torch.long,
        device=device,
    )
    finished = torch.zeros(
        len(questions), dtype=torch.bool, device=device
    )

    forbidden_ids = [
        tokenizer.token_to_id(token)
        for token in ["[PAD]", "[BOS]", "[UNK]"]
        if tokenizer.token_to_id(token) is not None
    ]

    for _ in range(max_new_tokens):

        logits = model.decode(
            generated,
            memory,
            memory_padding_mask=src_padding,
        )[:, -1]
        logits[:, forbidden_ids] = -float("inf")
        next_tokens = logits.argmax(dim=-1)

        next_tokens = next_tokens.masked_fill(finished, EOS_ID)
        generated = torch.cat(
            [generated, next_tokens.unsqueeze(1)],
            dim=1,
        )

        finished |= next_tokens.eq(EOS_ID)
        if finished.all().item():
            break

    answers = []
    for ids in generated[:, 1:].cpu().tolist():
        if EOS_ID in ids:
            ids = ids[:ids.index(EOS_ID)]
        answers.append(tokenizer.decode(ids, skip_special_tokens=True))

    return answers

In [ ]:
def generate(model, question, max_new_tokens=64):
    return generate_batch(
        model=model,
        tokenizer=tokenizer,
        questions=[question],
        max_src_len=model.max_src_len,
        max_new_tokens=max_new_tokens,
    )[0]


checkpoint = torch.load(
    output_dir / "best.pt", map_location=device, weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])

table = wandb.Table(columns=["question", "reference", "prediction"])

for example in valid_raw.select(range(min(5, len(valid_raw)))):
    prediction = generate(model, example["question"])
    table.add_data(example["question"], example["code"], prediction)
    print("Question:", example["question"])
    print("Prediction:\n", prediction, "\n")

run.log({"val/examples": table})
run.finish()

Question: Emily and her friend Sarah decided to start a lemonade stand. Emily made 20 cups of lemonade, which is 3/4 of the total amount of lemonade they have. Sarah made twice as many cups of lemonade as Emily and an additional 10 cups of orange juice. How much liquid in total do they have?
Prediction:
 def solve() -> int:
    # Step 1: Calculate the total number of lemonade in the total number of lemonade in the total number of lemonade in the question
    # Step 2: Calculate the number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total number of lemonade in the total
    lemonade_cups = 10
    lemonade_cups = lemonade_cups * 2/4
    lemonade_cups = lemonade_cups * 10
    lemonade_cups = lemonade_cups * 10
    total_cups = lemonade_cups + lemonade_cups
    result = 

Question: In a garden, there are 2000 flowers. Every day, 300 flowers bloom, while 500 flowers wither away. How many f

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅██████████████
global_step,▁▁▁▁▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
train/epoch_loss,█▃▁
train/grad_norm,▃▄▁▄▃▃▃▃▃▃▅▄▃▃▄▅▄▃▃▆▇▅▄▆▄▅▄▅▅▄▅▆▅▅▃▄▅▄▅█
train/loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/target_tokens_per_second,▁▆█▆▂▅▅▇▅▅▅▆▅▄▄▃▅▄▄▅▆▄▄▄▄▄▃▅▅▃▄▄▂▆▅▅▃▅▄▃
val/loss,█▃▁
val/token_accuracy,▁▆█
best/epoch,3
best/val_loss,1.15746


In [ ]:
hf_api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(output_dir),
    path_in_repo=f"runs/{run.id}",
)

CommitInfo(commit_url='https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer/commit/7e71a4df79099aec224e46aa404529e8e7033d55', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7e71a4df79099aec224e46aa404529e8e7033d55', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MrZVIL/hw2-cg-baseline_transformer', endpoint='https://huggingface.co', repo_type='model', repo_id='MrZVIL/hw2-cg-baseline_transformer'), pr_revision=None, pr_num=None)

In [ ]:
def test_collate_fn(examples):
    return {"question": [example["question"] for example in examples]}


test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=test_collate_fn,
    num_workers=0,
)

In [ ]:
with open("/kaggle/working/submission_batch.jsonl", "w", encoding="utf-8") as f:
    for tasks in tqdm(test_loader, desc="Generating"):
        with amp_context():
            codes = generate_batch(
                model=model,
                tokenizer=tokenizer,
                questions=tasks["question"],
                max_src_len=config["max_src_len"],
                max_new_tokens=64,
            )
        for code in codes:
            f.write(json.dumps({"code": code}, ensure_ascii=False) + "\n")

Generating:   0%|          | 0/60 [00:00<?, ?it/s]